In [0]:
%sql
-- Data Cube: Order-grain table with ONLY KPI-relevant columns
-- Grain: 1 row per order_id | Columns: KPI-mapped only

CREATE OR REPLACE TABLE workspace.cube.data_cube AS

WITH order_invoice AS (
    SELECT
        order_id,
        SUM(invoice_amount) AS invoice_amount,
        DATE_FORMAT(MIN(invoice_date), 'yyyy-MM') AS invoice_month
    FROM workspace.cube.fact_invoice
    GROUP BY order_id
),

order_estimate AS (
    SELECT
        order_id,
        MIN(CASE WHEN version_no = 1 THEN estimate_amount END) AS initial_estimate,
        MAX_BY(estimate_amount, version_no) AS final_estimate,
        MAX_BY(estimator_id, version_no) AS estimator_id
    FROM workspace.cube.fact_estimate
    WHERE estimate_amount IS NOT NULL
    GROUP BY order_id
)

SELECT
    -- === ORDER CORE (All KPIs) ===
    fo.order_id,
    fo.order_status,
    fo.service_type,
    YEAR(fo.vehicle_in_datetime) AS order_year,
    DATE_FORMAT(fo.vehicle_in_datetime, 'yyyy-MM') AS order_month,

    -- === STORE DIMENSION (KPIs 1,2,3,4,5,6,7,8) ===
    ds.store_id,
    ds.store_name,
    ds.manager_name,

    -- === TECHNICIAN DIMENSION (KPIs 6, 10) ===
    fo.technician_id,
    dt.technician_name,

    -- === DAYS IN SHOP (KPIs 2, 10) ===
    ROUND(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.vehicle_out_datetime) / 24.0, 2) AS days_in_shop,

    -- === STAGE CYCLE TIMES (KPI 8) ===
    ROUND(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.actual_work_start_datetime) / 24.0, 2) AS days_intake_to_start,
    ROUND(TIMESTAMPDIFF(HOUR, fo.actual_work_start_datetime, fo.actual_completion_datetime) / 24.0, 2) AS days_start_to_completion,
    ROUND(TIMESTAMPDIFF(HOUR, fo.actual_completion_datetime, fo.actual_delivery_datetime) / 24.0, 2) AS days_completion_to_delivery,
    ROUND(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.actual_delivery_datetime) / 24.0, 2) AS total_cycle_days,

    -- === COMPLETION ACCURACY (KPI 6) ===
    CASE
        WHEN fo.actual_completion_datetime IS NOT NULL AND fo.planned_completion_datetime IS NOT NULL
            THEN CASE WHEN fo.actual_completion_datetime <= fo.planned_completion_datetime THEN TRUE ELSE FALSE END
        ELSE NULL
    END AS is_on_time_completion,
    ROUND(TIMESTAMPDIFF(HOUR, fo.planned_completion_datetime, fo.actual_completion_datetime) / 24.0, 2) AS completion_variance_days,

    -- === REVENUE (KPIs 1, 5, 7) ===
    oi.invoice_amount,
    oi.invoice_month,

    -- === SURVEY (KPIs 3, 4) ===
    fs.survey_id,
    fs.responded_flag,
    fs.delivered_on_time_rating,
    fs.work_quality_rating,
    fs.cleanliness_rating,
    fs.communication_rating,
    fs.overall_satisfaction_rating,

    -- === ESTIMATE (KPI 9) ===
    oe.estimator_id,
    de.estimator_name,
    oe.initial_estimate,
    oe.final_estimate,
    ROUND(
        CASE WHEN oe.initial_estimate > 0
            THEN ABS(oe.final_estimate - oe.initial_estimate) * 100.0 / oe.initial_estimate
            ELSE NULL
        END, 2
    ) AS estimate_deviation_pct,
    ROUND(
        CASE WHEN oe.initial_estimate > 0
            THEN 100 - (ABS(oe.final_estimate - oe.initial_estimate) * 100.0 / oe.initial_estimate)
            ELSE NULL
        END, 2
    ) AS estimate_accuracy_pct,

    -- === BUDGET (KPI 5) ===
    fb.budget_amount

FROM workspace.cube.fact_order fo
JOIN workspace.cube.dim_store ds ON fo.store_id = ds.store_id
LEFT JOIN workspace.cube.dim_technician dt ON fo.technician_id = dt.technician_id
LEFT JOIN order_invoice oi ON fo.order_id = oi.order_id
LEFT JOIN workspace.cube.fact_survey fs ON fo.order_id = fs.order_id
LEFT JOIN order_estimate oe ON fo.order_id = oe.order_id
LEFT JOIN workspace.cube.dim_estimator de ON oe.estimator_id = de.estimator_id
LEFT JOIN workspace.cube.fact_budget fb
    ON fo.store_id = fb.ns_store_id
    AND DATE_FORMAT(fo.vehicle_in_datetime, 'yyyy-MM') = fb.month

In [0]:
%sql
-- Verify: Row count, column count, order status breakdown
SELECT
    (SELECT COUNT(*) FROM workspace.cube.data_cube) AS total_rows,
    (SELECT COUNT(DISTINCT order_id) FROM workspace.cube.data_cube) AS distinct_orders,
    (SELECT COUNT(*) FROM workspace.cube.fact_order) AS fact_order_rows
;

SELECT order_status, COUNT(*) AS cnt,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.cube.data_cube), 1) AS pct
FROM workspace.cube.data_cube
GROUP BY order_status ORDER BY cnt DESC

In [0]:
%sql
-- Verify: KPI measure coverage (non-null %)
SELECT
    COUNT(*) AS total,
    ROUND(COUNT(days_in_shop) * 100.0 / COUNT(*), 1) AS pct_days_in_shop,
    ROUND(COUNT(total_cycle_days) * 100.0 / COUNT(*), 1) AS pct_cycle_times,
    ROUND(COUNT(is_on_time_completion) * 100.0 / COUNT(*), 1) AS pct_on_time_flag,
    ROUND(COUNT(invoice_amount) * 100.0 / COUNT(*), 1) AS pct_invoice,
    ROUND(COUNT(survey_id) * 100.0 / COUNT(*), 1) AS pct_survey,
    ROUND(COUNT(estimate_accuracy_pct) * 100.0 / COUNT(*), 1) AS pct_estimate,
    ROUND(COUNT(budget_amount) * 100.0 / COUNT(*), 1) AS pct_budget
FROM workspace.cube.data_cube

In [0]:
%sql
-- Sample: Preview cube with key columns
SELECT order_id, order_status, service_type, order_year, order_month,
    store_name, manager_name, technician_name,
    days_in_shop, total_cycle_days, is_on_time_completion,
    invoice_amount, invoice_month,
    responded_flag, overall_satisfaction_rating,
    estimator_name, estimate_accuracy_pct,
    budget_amount
FROM workspace.cube.data_cube
WHERE order_status = 'COMPLETED'
limit 100